# 2-6.1.1 dictionary 사용

In [ ]:
from nltk.tokenize import sent_tokenize
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

In [ ]:
raw_text = "A barber is a person. a barber is good person. " \
"a barber is huge person. he Knew A Secret! " \
"The Secret He Kept is huge secret. Huge secret. " \
"His barber kept his word. a barber kept his word. " \
"His barber kept his secret. But keeping and keeping " \
"such a huge secret to himself was driving the barber crazy. " \
"the barber went up a huge mountain."

# 문장 토큰화
sentences = sent_tokenize(raw_text)
print(sentences)


텍스트 데이터를 문장 단위로 토큰화하였다.

In [ ]:
vocab = {}
preprocessed_sentences = []
stop_words = set(stopwords.words('english'))

for sentence in sentences:
    # 단어 토큰화
    tokenized_sentence = word_tokenize(sentence)
    result = []

    for word in tokenized_sentence: 
        word = word.lower() # 모든 단어를 소문자화하여 단어의 개수를 줄인다.
        if word not in stop_words: # 단어 토큰화 된 결과에 대해서 불용어를 제거한다.
            if len(word) > 2: # 단어 길이가 2이하인 경우에 대하여 추가로 단어를 제거한다.
                result.append(word)
                if word not in vocab:
                    vocab[word] = 0 
                vocab[word] += 1
    preprocessed_sentences.append(result) 
print(preprocessed_sentences)


정제 작업과 정규화 작업을 병행하며, 단어 토큰화를 수행한다.  
위 코드에서는 단어들에 lower시켜서 단어의 종류를 통일시키고, 불용어와 단어 길이가 2이하인 단어를 제외시켰다.  
그리하여 문장 토큰화 데이터별로 어떤 단어 토큰을 가지고있는지 알게되었다.

In [ ]:
print('단어 집합 :',vocab)


In [ ]:
# 'barber'라는 단어의 빈도수 출력
print(vocab["barber"])


vocab에는 각 단어의 빈도수가 dictionary로 기록되어있고, key값을 이용해서 빈도 수를 찾을 수 있다.

In [ ]:
vocab_sorted = sorted(vocab.items(), key = lambda x:x[1], reverse = True)
print(vocab_sorted)


빈도수를 내림차순으로 정렬하였다.

In [ ]:
word_to_index = {}
i = 0
for (word, frequency) in vocab_sorted :
    if frequency > 1 : # 빈도수가 작은 단어는 제외.
        i = i + 1
        word_to_index[word] = i

print(word_to_index)


빈도수가 높은 순서대로 1부터 index를 부여한다.  
이 작업과 동시에 빈도수가 낮은 단어를 제외시킨다.

In [ ]:
vocab_size = 5

# 인덱스가 5 초과인 단어 제거
words_frequency = [word for word, index in word_to_index.items() if index >= vocab_size + 1]

# 해당 단어에 대한 인덱스 정보를 삭제
for w in words_frequency:
    del word_to_index[w]
print(word_to_index)


자연어 처리에서 모든 단어를 사용하기 보다는 빈도수가 높은 n개의 단어만 사용하는 경우가 많다.  
그리하여 위 코드에서는 빈도수가 높은 5개의 단어만 사용한다고 정의한다.

In [ ]:
word_to_index['OOV'] = len(word_to_index) + 1
print(word_to_index)


이제 단어 토큰들을 각 정수에 대입하는 작업을 해야한다.  
그런데 첫번째 문장의 ['barber', 'person']은 [1,5]로 인코딩이 가능하지만, 두번째 문장의 ['barber', 'good', 'person']는 word_to_index에 포함되지 않은 'good'이라는 단어를 가지고있다. 
   
이러한 단어 집합에 존재하지않는 단어들이 생기는 상황을 Out-Of-Vocabulary 문제라고 한다.(약어로 'OOV 문제'라고도 한다.)  
이러한 문제를 해결하기위해 단어 집합에 'OOV'란 단어를 추가하여서 포함되지않는 단어들은 'OOV'로 인코딩한다.

In [ ]:
encoded_sentences = []
for sentence in preprocessed_sentences:
    encoded_sentence = []
    for word in sentence:
        try:
            # 단어 집합에 있는 단어라면 해당 단어의 정수를 리턴.
            encoded_sentence.append(word_to_index[word])
        except KeyError:
            # 만약 단어 집합에 없는 단어라면 'OOV'의 정수를 리턴.
            encoded_sentence.append(word_to_index['OOV'])
    encoded_sentences.append(encoded_sentence)
print(encoded_sentences)


이를통해 파이썬의 기본 구조인 dictionary를 사용한 정수 인코딩을 실습해보았다.

# 2-6.1.2 Counter 사용

In [ ]:
from collections import Counter


In [ ]:
print(preprocessed_sentences)


'dictionary 사용'에서 사용한 단어 토큰화 결과를 사용한다.

In [ ]:
# words = np.hstack(preprocessed_sentences)으로도 수행 가능.
all_words_list = sum(preprocessed_sentences, [])
print(all_words_list)


원래 sentences는 문장별로 분리해 단어 토큰을 모아두었기때문에 Counter를 쓰기위해 하나의 리스트로 만든다.

In [ ]:
# 파이썬의 Counter 모듈을 이용하여 단어의 빈도수 카운트
vocab = Counter(all_words_list)
print(vocab)


파이썬의 Counter를 사용하여 단어의 빈도수를 dictionary를 사용할때보다 훨씬 편하게 기록한다.

In [ ]:
print(vocab["barber"]) # 'barber'라는 단어의 빈도수 출력


In [ ]:
vocab_size = 5
vocab = vocab.most_common(vocab_size) # 등장 빈도수가 높은 상위 5개의 단어만 저장
vocab


most_common(n)명령어를 통해서 n개의 상위 빈도수 단어들을 내림차순으로 저장한다.

In [ ]:
word_to_index = {}
i = 0
for (word, frequency) in vocab :
    i = i + 1
    word_to_index[word] = i

print(word_to_index)


'dictionary 사용'처럼 높은 빈도수의 단어부터 정수 인덱스를 부여한다.

# 2-6.1.3 NLTK의 FreqDist 사용

In [ ]:
from nltk import FreqDist
import numpy as np


In [ ]:
# np.hstack으로 문장 구분을 제거
vocab = FreqDist(np.hstack(preprocessed_sentences))


In [ ]:
vocab_size = 5
vocab = vocab.most_common(vocab_size) # 등장 빈도수가 높은 상위 5개의 단어만 저장
print(vocab)

numpy로 문장 구분 배열을 합친 후 FreqDist로 빈도수를 저장한다.  
그런데 막상 결과를 보면 'Counter 사용'과는 무언가 다른 결과를 보여준다.

In [ ]:
vocab = FreqDist([str(w) for w in np.hstack(preprocessed_sentences)])


사실 np.hstack()을 실행하면 원래 str 원소를 np.str_로 바꾸는 과정이 존재하는데, numpy 2.0부터는 np.str_처럼 타입을 명시하도록 변경되었다.  
그래서 위의 코드로 수정을 해야 'Counter 사용'과 완전히 같은 결과를 얻을 수 있다.

In [ ]:
vocab_size = 5
vocab = vocab.most_common(vocab_size) # 등장 빈도수가 높은 상위 5개의 단어만 저장
print(vocab)


상위 5개의 단어 집합을 저장한다.

In [ ]:
word_to_index = {word[0] : index + 1 for index, word in enumerate(vocab)}
print(word_to_index)


# 2-6.1.4 enumerate 이해하기

In [ ]:
test_input = ['a', 'b', 'c', 'd', 'e']
for index, value in enumerate(test_input): # 입력의 순서대로 0부터 인덱스를 부여함.
  print("value : {}, index: {}".format(value, index))


리스트의 각 토큰에 인덱스가 순차적으로 부여된 모습을 보인다.

# 2-6.2 Keras의 텍스트 전처리

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer


In [ ]:
preprocessed_sentences = [['barber', 'person'], ['barber', 'good', 'person'], ['barber', 'huge', 'person'], ['knew', 'secret'], ['secret', 'kept', 'huge', 'secret'], ['huge', 'secret'], ['barber', 'kept', 'word'], ['barber', 'kept', 'word'], ['barber', 'kept', 'secret'], ['keeping', 'keeping', 'huge', 'secret', 'driving', 'barber', 'crazy'], ['barber', 'went', 'huge', 'mountain']]

tokenizer = Tokenizer()

# fit_on_texts()안에 코퍼스를 입력으로 하면 빈도수를 기준으로 단어 집합을 생성.
tokenizer.fit_on_texts(preprocessed_sentences) 


In [ ]:
print(tokenizer.word_index)


In [ ]:
print(tokenizer.word_counts)


keras의 tokenizer를 사용하여 더 간편하게 빈도수 순 인덱스 부여를 완료하였다.

In [ ]:
print(tokenizer.texts_to_sequences(preprocessed_sentences))


토큰 인코딩까지 각각 한줄의 코드들로 끝난다.

In [ ]:
vocab_size = 5
tokenizer = Tokenizer(num_words = vocab_size + 1) # 상위 5개 단어만 사용
tokenizer.fit_on_texts(preprocessed_sentences)


num_words애 1을 더하는 이유는 num_words는 인덱스 0부터 카운트하는데, 단어 집합의 인덱스는 1부터 시작하기때문에 더한다.

In [ ]:
print(tokenizer.word_index)


In [ ]:
print(tokenizer.word_counts)


In [ ]:
print(tokenizer.texts_to_sequences(preprocessed_sentences))


상위 5개의 단어만 집합에 유지시켰지만 word_index와counts를 출력하면 13개의 단어가 모두 출력된다.
하지만 text_to_sequences를 사용할 때는 정상적으로 5개의 단어만 존재하도록 잘 적용된 것을 볼수있다.

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(preprocessed_sentences)


In [ ]:
vocab_size = 5
words_frequency = [word for word, index in tokenizer.word_index.items() if index >= vocab_size + 1] 

# 인덱스가 5 초과인 단어 제거
for word in words_frequency:
    del tokenizer.word_index[word] # 해당 단어에 대한 인덱스 정보를 삭제
    del tokenizer.word_counts[word] # 해당 단어에 대한 카운트 정보를 삭제

print(tokenizer.word_index)
print(tokenizer.word_counts)
print(tokenizer.texts_to_sequences(preprocessed_sentences))


만약 word_indes나 counts에서도 5개의 단어만 출력시키고싶다면 위의 코드처럼 del로 조건을 벗어나는 단어는 집합에서 삭제하는 방법이 존재한다.

In [ ]:
# 숫자 0과 OOV를 고려해서 단어 집합의 크기는 +2
vocab_size = 5
tokenizer = Tokenizer(num_words = vocab_size + 2, oov_token = 'OOV')
tokenizer.fit_on_texts(preprocessed_sentences)


In [ ]:
print('단어 OOV의 인덱스 : {}'.format(tokenizer.word_index['OOV']))


In [ ]:
print(tokenizer.texts_to_sequences(preprocessed_sentences))


Keras의 Tokenizer는 기본적으로 OOV에 해당하는 단어를 정수 변환하지않고 제거한다는 특징을 가진다.
그렇기때문에 OOV 단어를 보존하고싶으면 oov_token 인자를 사용하여 index를 부여하고 인코딩하는 방법이 존재한다.